# Notebook 04 — ARMA-to-CARMA Initial Parameter Mapping

Converts discrete ARMA$(p,q)$ coefficients to initial CARMA$(p,q)$ parameters. AR eigenvalues $\lambda_i$ are computed from the roots $r_i$ of the AR polynomial via $\lambda_i = \ln(r_i)/h$, giving AR coefficients $a_1,\ldots,a_p$ as elementary symmetric polynomials of $\lambda_i$.

- **Temperature — CARMA$(2,1)$:** The spectral-density discriminant is numerically zero at this parameter point, so $b_0$ cannot be resolved analytically. MA parameters are instead set directly ($b_0 = 1$, $b_1 \approx 0.57$) from the hourly-scale ARMA estimate and confirmed by QMLE in notebook 06.
- **Log-price — CARMA$(3,2)$:** $b_0$ is solved from the spectral-density matching identity; $b_1$ is initialized from the ARMA MA coefficient. These feed the QMLE starting point for notebook 07.

**Inputs:** ARMA coefficients from notebook 03  
**Outputs:** none (initial values passed manually to notebooks 06/07)

### CARMA(2,1) — temperature

Converts ARMA$(2,1)$ coefficients (temperature) to CARMA$(2,1)$ AR parameters via eigenvalue decomposition of the AR polynomial, then attempts to solve for $b_0$ via the spectral-density matching formula. The discriminant is numerically zero at this parameter point, so $b_0$ cannot be resolved analytically here; the initial MA parameters for R are instead set directly ($b_0=1, b_1\approx 0.57$) based on the hourly-scale ARMA estimate.

In [ ]:
import numpy as np
from scipy.linalg import expm

phi1 =1.3758
phi2 = -0.3921
theta=-0.130440722070374

phi=np.array([phi1,phi2])
print("arma coefficients :", phi)

if phi2>0:
    print("problem: phi2 must be <0")

coeffarma = [-phi2, -phi1, 1]

roots = np.roots(coeffarma)
print("roots :", roots)

lambda_vals = -np.log(roots)*8760
print("lambda :", lambda_vals)

def coeffcarma(eigenvalues):
    [lam1,lam2] = eigenvalues.copy()
    a1=-(lam1+lam2)
    a2=lam1*lam2
    return [a1,a2]

a=coeffcarma(lambda_vals)
a= np.array([float(x) for x in a])

if phi1-2*np.exp(-a[0]/2)<0:
    print("problem: phi1 must be >= 2*exp(-a1)/2")

print("carma AR coefficients :", a)

print("arma MA coefficients :", theta)

import numpy as np

def buildcompanion(a):
    p = len(a)
    A = np.zeros((p,p))
    for i in range(p-1):
        A[i,i+1] = 1

    A[p-1,:] = -np.array(a[::-1])
    return A

buildcompanion(a)

import numpy as np
from scipy.linalg import solve_continuous_lyapunov

def buildsigma(coefcarma):
    A=buildcompanion(coefcarma)
    p=len(coefcarma)
    ep=np.zeros((p,1))
    ep[p-1,0]=1

    Q=ep@ep.T

    sigma=solve_continuous_lyapunov(A,-Q)
    return sigma
buildsigma(a)

def m0(phi,a):
    A=buildcompanion(a)
    [phi1,phi2]=phi
    sigma=buildsigma(a)
    M0=((1+phi1**2+phi2**2)*np.identity(2)+(2*phi1*phi2-2*phi1)*expm(A)-2*phi2*expm(2*A))@sigma
    return M0

def m1(phi,a):
    A=buildcompanion(a)
    [phi1,phi2]=phi
    sigma=buildsigma(a)
    M1=((-phi1+phi1*phi2)*np.identity(2)+(1-phi2+phi1**2+phi2**2)*expm(A)+(-phi1+phi1*phi2)*expm(2*A)-phi2*expm(3*A))@sigma
    return M1

def m(phi,a,theta):
    M0=m0(phi,a)
    M1=m1(phi,a)
    return M0-((1+theta**2)/theta)*M1

M=m(phi,a,theta)

b0=(1/(2*M[0,0]))*(-(M[0,1]+M[1,0])+np.sqrt((M[0,1]+M[1,0])**2-4*M[0,0]*M[1,1]))

print((M[0,1]+M[1,0])**2-4*M[0,0]*M[1,1])
print("carma MA coefficient :", b0)
print("carma AR coefficients :", a)


arma coefficients : [ 1.37582643 -0.39218534]
roots : [2.47991716 1.02818549]
lambda : [-7956.05237966  -243.48934726]
carma AR coefficients : [   8199.54172692 1937214.00067306]
arma MA coefficients : -0.130440722070374
-1.0849398911186413e-12
carma MA coefficient : nan
carma AR coefficients : [   8199.54172692 1937214.00067306]


C:\Users\gabri\AppData\Local\Temp\ipykernel_16816\485097521.py:87: RuntimeWarning: invalid value encountered in sqrt
  b0=(1/(2*M[0,0]))*(-(M[0,1]+M[1,0])+np.sqrt((M[0,1]+M[1,0])**2-4*M[0,0]*M[1,1]))


### CARMA(3,2) — log-price

Converts ARMA$(3,2)$ coefficients (log-price) to CARMA$(3,2)$ AR and MA parameters. The AR roots of the ARMA polynomial include a complex conjugate pair, so the CARMA eigenvalues are complex; only their real parts enter the companion matrix $A$. The MA coefficient $b_0$ is real and well-defined here; $b_1$ and $b_2$ are initialized separately in notebook 07.

In [1]:
import numpy as np
from scipy.linalg import expm

phi = [1.7059 , -0.8198, 0.0948]
theta = [-0.0168860731322791]

phi=np.array(phi)
theta=np.array(theta)
[phi1,phi2,phi3]=phi 
[theta1]=theta

print("arma coefficients :", phi)

coeffarma = [-phi3,-phi2, -phi1, 1]

roots = np.roots(coeffarma)
print("roots :", roots)

lambda_vals = -np.log(roots)
print("lambda :", lambda_vals)

def coeffcarma(eigenvalues):
    [lam1,lam2,lam3] = eigenvalues.copy()
    a1=-(lam1+lam2+lam3)
    a2=lam1*lam2+lam2*lam3+lam1*lam3
    a3=-lam1*lam2*lam3
    return [a1,a2,a3]

a=coeffcarma(lambda_vals)
a= np.array([float(x) for x in a])

print("carma AR coefficients :", a)

import numpy as np

def buildcompanion(a):
    p = len(a)
    A = np.zeros((p,p))
    for i in range(p-1):
        A[i,i+1] = 1

    A[p-1,:] = -np.array(a[::-1])
    return A

buildcompanion(a)

import numpy as np
from scipy.linalg import solve_continuous_lyapunov

def buildsigma(coefcarma):
    A=buildcompanion(coefcarma)
    p=len(coefcarma)
    ep=np.zeros((p,1))
    ep[p-1,0]=1

    Q=ep@ep.T

    sigma=solve_continuous_lyapunov(A,-Q)
    return sigma
buildsigma(a)

def m0(phi,a):
    A=buildcompanion(a)
    [phi1,phi2,phi3]=phi
    sigma=buildsigma(a)
    M0=((1+phi1**2+phi2**2+phi3**2)*np.identity(3)+(-2*phi1+2*phi1*phi2+2*phi2*phi3)*expm(A)+(-2*phi2+2*phi1*phi3)*expm(2*A)+(-2*phi3)*expm(3*A))@sigma
    return M0

def m1(phi,a):
    A=buildcompanion(a)
    [phi1,phi2,phi3]=phi
    sigma=buildsigma(a)
    M1=((-phi1+phi1*phi2+phi2*phi3)*np.identity(3)+(1-phi2+phi1**2+phi2**2+phi1*phi3+phi3**2)*expm(A)+(-phi1+phi1*phi2+phi2*phi3-phi3)*expm(2*A)+(-phi2+phi1*phi3)*expm(3*A)+(-phi3)*expm(4*A))@sigma
    return M1

def m(phi,a,theta):
    M0=m0(phi,a)
    M1=m1(phi,a)
    return M0-((1+theta**2)/theta)*M1

M=m(phi,a,theta)
b0=(1/(2*M[0,0]))*(-(M[0,1]+M[1,0])+np.sqrt((M[0,1]+M[1,0])**2-4*M[0,0]*M[1,1]))

print((1/(2*M[0,0]))*(-(M[0,1]+M[1,0])))
print((M[0,1]+M[1,0])**2-4*M[0,0]*M[1,1])

print("carma MA coefficient :", b0)
print("carma AR coefficients :", a)

arma coefficients : [ 1.7059 -0.8198  0.0948]
roots : [5.90129248 1.68650869 1.05987815]
lambda : [-1.77517139 -0.52266053 -0.05815395]
carma AR coefficients : [2.35598587 1.06144002 0.05395593]
-2.727046182514233e-14
18.6081301556799
carma MA coefficient : 1.3244746288428033
carma AR coefficients : [2.35598587 1.06144002 0.05395593]


#### CARMA coefficient derivation — annual time-step ($h = 1/8760$ yr)

Converts the ARMA$(3,2)$ roots estimated at hourly frequency to CARMA$(3,2)$ parameters expressed in yr⁻¹. Each discrete AR root $r_i$ maps to a continuous eigenvalue via
$$\lambda_i = \frac{\ln r_i}{h}, \qquad h = \tfrac{1}{8760}\text{ yr}.$$
The AR polynomial coefficients $\{a_k\}$ of the companion matrix $A$ are recovered from the elementary symmetric polynomials of $\{\lambda_i\}$. The MA polynomial for CARMA$(3,2)$ has degree 2: $b(z) = b_0 + b_1 z + b_2 z^2$. Here $b_0$ is initialized from the spectral-density identity; $b_1$ and $b_2$ are set to initial values and refined by QMLE in notebook 07.

In [ ]:
import numpy as np
from scipy.linalg import expm, solve_continuous_lyapunov

# ARMA(3,2) estimated coefficients for log-price
phi   = np.array([1.5019, -0.5348, 0.0038])
theta = np.array([0.115685874183687])
dt    = 1 / 8760  # time step in years (1 hour)

# Step 1 — roots of the discrete AR polynomial
# z^3 - phi1*z^2 - phi2*z - phi3 = 0
coef_ar = np.array([1, -phi[0], -phi[1], -phi[2]])
roots   = np.roots(coef_ar)
print("Discrete AR roots:", roots)
print("Moduli:           ", np.abs(roots))

# Step 2 — CARMA eigenvalues in yr^-1
# Stationarity of the ARMA model requires |r_i| < 1;
# eigenvalues are lambda_i = log(r_i) / dt
lambda_vals = np.log(roots) / dt
print("CARMA eigenvalues (yr^-1):", lambda_vals)
print("Half-lives (hours):        ", np.log(2) / np.abs(lambda_vals) * 8760)

# Step 3 — AR coefficients of CARMA from elementary symmetric polynomials of eigenvalues
a      = np.real(np.poly(lambda_vals))   # [1, a1, a2, a3]
a_carma = a[1:]                           # [a1, a2, a3] in yr^-1
print("CARMA AR coefficients (yr^-1, yr^-2, yr^-3):", a_carma)

# Step 4 — companion matrix and eigenvalue check
def build_companion(a_carma):
    p = len(a_carma)
    A = np.zeros((p, p))
    for i in range(p - 1):
        A[i, i + 1] = 1.0
    A[p - 1, :] = -a_carma[::-1]
    return A

A = build_companion(a_carma)
print("Eigenvalues of A:", np.linalg.eigvals(A))

# Step 5 — initial MA coefficients (CARMA(3,1): b0 only; b1 set to 0)
b0_init = 1.0
b1_init = 0.0
print(f"Initial MA parameters: b0 = {b0_init},  b1 = {b1_init}")